In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt

# Load the dataset and prepare the data
file_path = '../../data/Data3.csv'  # Update with your actual file path
data = pd.read_csv(file_path)
data['Site'] = data['ERBS'].str[:7]
data['STARTTIME_DATE'] = pd.to_datetime(data['STARTTIME_DATE'])
grouped_data = data.groupby(['EUTRANCELLFDD', 'STARTTIME_DATE']).mean().reset_index()

# Scaling function
scaler = MinMaxScaler()

# Define the create_sequences function
def create_sequences(data, seq_length):
    xs, ys = [], []
    for i in range(len(data) - seq_length):
        x = data[i:i+seq_length]
        y = data[i+seq_length]
        xs.append(x)
        ys.append(y)
    return np.array(xs), np.array(ys)

# Function to calculate MSE for each EUTRANCELLFDD
def calculate_mse_for_eutrancell(target_column):
    mse_results = {}
    eutrancell_predictions = {}
    
    for eutrancell in grouped_data['EUTRANCELLFDD'].unique():
        eutrancell_data = grouped_data[grouped_data['EUTRANCELLFDD'] == eutrancell][target_column].values.reshape(-1, 1)
        
        if len(eutrancell_data) <= 10:
            print(f"Skipping {eutrancell} due to insufficient data.")
            continue

        # Scale data
        eutrancell_data_scaled = scaler.fit_transform(eutrancell_data)
        
        # Create sequences
        X, y = create_sequences(eutrancell_data_scaled, seq_length=10)
        
        if len(X) == 0 or len(X) < 2:  # If not enough sequences, skip the cell
            print(f"Skipping {eutrancell} due to insufficient sequences.")
            continue
        
        # If we have less than 10 samples, reduce validation_split or remove it
        if len(X) < 16:  # Less than 16 samples, 80% of it is less than 10
            validation_split = 0.0
        else:
            validation_split = 0.2
        
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

        # Define and train the LSTM model
        model = Sequential()
        model.add(LSTM(50, return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])))
        model.add(Dropout(0.2))
        model.add(LSTM(50, return_sequences=False))
        model.add(Dropout(0.2))
        model.add(Dense(1))
        
        model.compile(optimizer=Adam(learning_rate=0.001), loss='mean_squared_error')
        
        # Fit the model without validation if not enough data
        model.fit(X_train, y_train, epochs=50, batch_size=16, validation_split=validation_split, verbose=1)
        
        # Predict
        y_pred = model.predict(X_test)
        
        # Inverse transform the predictions and true values before calculating MSE
        y_test_inverse = scaler.inverse_transform(y_test)
        y_pred_inverse = scaler.inverse_transform(y_pred)
        
        # Calculate MSE on the inverse-transformed data
        mse = mean_squared_error(y_test_inverse, y_pred_inverse)
        
        mse_results[eutrancell] = mse
        eutrancell_predictions[eutrancell] = (y_test_inverse, y_pred_inverse)
    
    return mse_results, eutrancell_predictions

# Calculate MSE and predictions for DL_TRAFFIC_MB
mse_results_traffic, eutrancell_predictions_traffic = calculate_mse_for_eutrancell('DL_TRAFFIC_MB')

# Sort results
sorted_mse_results = sorted(mse_results_traffic.items(), key=lambda x: x[1])
best_5_eutrancell = sorted_mse_results[:5]
worst_5_eutrancell = sorted_mse_results[-5:]

# Plot actual vs predicted values
def plot_actual_vs_predicted(eutrancell, y_test, y_pred):
    plt.figure(figsize=(10, 5))
    plt.plot(y_test, label='Actual')
    plt.plot(y_pred, label='Predicted')
    plt.title(f'Actual vs Predicted for DL_TRAFFIC_MB - EUTRANCELLFDD: {eutrancell}')
    plt.xlabel('Time Steps')
    plt.ylabel('DL_TRAFFIC_MB')
    plt.legend()
    plt.show()

# Plot for best and worst 5 cells
print("Best 5 EUTRANCELLFDD based on MSE:", best_5_eutrancell)
for eutrancell, _ in best_5_eutrancell:
    if eutrancell in eutrancell_predictions_traffic:
        y_test, y_pred = eutrancell_predictions_traffic[eutrancell]
        plot_actual_vs_predicted(eutrancell, y_test, y_pred)

print("Worst 5 EUTRANCELLFDD based on MSE:", worst_5_eutrancell)
for eutrancell, _ in worst_5_eutrancell:
    if eutrancell in eutrancell_predictions_traffic:
        y_test, y_pred = eutrancell_predictions_traffic[eutrancell]
        plot_actual_vs_predicted(eutrancell, y_test, y_pred)

C:\Users\HP\AppData\Local\Temp\ipykernel_22160\1863103781.py:16: FutureWarning: The default value of numeric_only in DataFrameGroupBy.mean is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  grouped_data = data.groupby(['EUTRANCELLFDD', 'STARTTIME_DATE']).mean().reset_index()


Epoch 1/50
15/15 [==============================] - 15s 284ms/step - loss: 0.2086 - val_loss: 0.0252
Epoch 2/50
15/15 [==============================] - 0s 25ms/step - loss: 0.0301 - val_loss: 0.0286
Epoch 3/50
15/15 [==============================] - 0s 24ms/step - loss: 0.0165 - val_loss: 0.0139
Epoch 4/50
15/15 [==============================] - 0s 24ms/step - loss: 0.0178 - val_loss: 0.0179
Epoch 5/50
15/15 [==============================] - 0s 23ms/step - loss: 0.0124 - val_loss: 0.0116
Epoch 6/50
15/15 [==============================] - 0s 16ms/step - loss: 0.0119 - val_loss: 0.0135
Epoch 7/50
15/15 [==============================] - 0s 19ms/step - loss: 0.0130 - val_loss: 0.0108
Epoch 8/50
15/15 [==============================] - 0s 13ms/step - loss: 0.0122 - val_loss: 0.0125
Epoch 9/50
15/15 [==============================] - 0s 14ms/step - loss: 0.0123 - val_loss: 0.0112
Epoch 10/50
15/15 [==============================] - 0s 13ms/step - loss: 0.0105 - val_loss: 0.0108
Epoch 1

15/15 [==============================] - 0s 14ms/step - loss: 0.0079 - val_loss: 0.0082
Epoch 33/50
15/15 [==============================] - 0s 12ms/step - loss: 0.0078 - val_loss: 0.0087
Epoch 34/50
15/15 [==============================] - 0s 13ms/step - loss: 0.0086 - val_loss: 0.0107
Epoch 35/50
15/15 [==============================] - 0s 12ms/step - loss: 0.0072 - val_loss: 0.0083
Epoch 36/50
15/15 [==============================] - 0s 13ms/step - loss: 0.0071 - val_loss: 0.0090
Epoch 37/50
15/15 [==============================] - 0s 13ms/step - loss: 0.0081 - val_loss: 0.0089
Epoch 38/50
15/15 [==============================] - 0s 15ms/step - loss: 0.0079 - val_loss: 0.0091
Epoch 39/50
15/15 [==============================] - 0s 13ms/step - loss: 0.0078 - val_loss: 0.0089
Epoch 40/50
15/15 [==============================] - 0s 14ms/step - loss: 0.0088 - val_loss: 0.0080
Epoch 41/50
15/15 [==============================] - 0s 14ms/step - loss: 0.0073 - val_loss: 0.0081
Epoch 42/50


15/15 [==============================] - 0s 17ms/step - loss: 0.0111 - val_loss: 0.0106
Epoch 14/50
15/15 [==============================] - 0s 15ms/step - loss: 0.0108 - val_loss: 0.0105
Epoch 15/50
15/15 [==============================] - 0s 17ms/step - loss: 0.0119 - val_loss: 0.0106
Epoch 16/50
15/15 [==============================] - 0s 17ms/step - loss: 0.0111 - val_loss: 0.0110
Epoch 17/50
15/15 [==============================] - 0s 13ms/step - loss: 0.0123 - val_loss: 0.0105
Epoch 18/50
15/15 [==============================] - 0s 14ms/step - loss: 0.0110 - val_loss: 0.0111
Epoch 19/50
15/15 [==============================] - 0s 13ms/step - loss: 0.0132 - val_loss: 0.0107
Epoch 20/50
15/15 [==============================] - 0s 14ms/step - loss: 0.0115 - val_loss: 0.0108
Epoch 21/50
15/15 [==============================] - 0s 14ms/step - loss: 0.0116 - val_loss: 0.0111
Epoch 22/50
15/15 [==============================] - 0s 14ms/step - loss: 0.0126 - val_loss: 0.0109
Epoch 23/50


15/15 [==============================] - 0s 22ms/step - loss: 0.0121 - val_loss: 0.0109
Epoch 45/50
15/15 [==============================] - 0s 18ms/step - loss: 0.0123 - val_loss: 0.0136
Epoch 46/50
15/15 [==============================] - 0s 22ms/step - loss: 0.0116 - val_loss: 0.0114
Epoch 47/50
15/15 [==============================] - 0s 30ms/step - loss: 0.0121 - val_loss: 0.0123
Epoch 48/50
15/15 [==============================] - 0s 27ms/step - loss: 0.0111 - val_loss: 0.0117
Epoch 49/50
15/15 [==============================] - 0s 25ms/step - loss: 0.0101 - val_loss: 0.0109
Epoch 50/50
3/3 [==============================] - 3s 12ms/step
Epoch 1/50
15/15 [==============================] - 16s 228ms/step - loss: 0.1756 - val_loss: 0.0505
Epoch 2/50
15/15 [==============================] - 0s 17ms/step - loss: 0.0200 - val_loss: 0.0159
Epoch 3/50
15/15 [==============================] - 0s 21ms/step - loss: 0.0122 - val_loss: 0.0098
Epoch 4/50
15/15 [==============================]

...

15/15 [==============================] - 0s 17ms/step - loss: 0.0135 - val_loss: 0.0126
Epoch 20/50
15/15 [==============================] - 0s 19ms/step - loss: 0.0133 - val_loss: 0.0079
Epoch 21/50
15/15 [==============================] - 0s 19ms/step - loss: 0.0115 - val_loss: 0.0072
Epoch 22/50
15/15 [==============================] - 0s 19ms/step - loss: 0.0113 - val_loss: 0.0075
Epoch 23/50
15/15 [==============================] - 0s 19ms/step - loss: 0.0118 - val_loss: 0.0084
Epoch 24/50
15/15 [==============================] - 0s 17ms/step - loss: 0.0126 - val_loss: 0.0118
Epoch 25/50
15/15 [==============================] - 0s 19ms/step - loss: 0.0143 - val_loss: 0.0081
Epoch 26/50
15/15 [==============================] - 0s 19ms/step - loss: 0.0115 - val_loss: 0.0073
Epoch 27/50
15/15 [==============================] - 0s 20ms/step - loss: 0.0116 - val_loss: 0.0072
Epoch 28/50
15/15 [==============================] - 0s 18ms/step - loss: 0.0117 - val_loss: 0.0072
Epoch 29/50


3/3 [==============================] - 2s 7ms/step
Epoch 1/50
14/14 [==============================] - 12s 203ms/step - loss: 0.0951 - val_loss: 0.0390
Epoch 2/50
14/14 [==============================] - 0s 18ms/step - loss: 0.0298 - val_loss: 0.0270
Epoch 3/50
14/14 [==============================] - 0s 20ms/step - loss: 0.0251 - val_loss: 0.0186
Epoch 4/50
14/14 [==============================] - 0s 19ms/step - loss: 0.0228 - val_loss: 0.0189
Epoch 5/50
14/14 [==============================] - 0s 19ms/step - loss: 0.0248 - val_loss: 0.0191
Epoch 6/50
14/14 [==============================] - 0s 20ms/step - loss: 0.0230 - val_loss: 0.0187
Epoch 7/50
14/14 [==============================] - 0s 20ms/step - loss: 0.0243 - val_loss: 0.0191
Epoch 8/50
14/14 [==============================] - 0s 26ms/step - loss: 0.0231 - val_loss: 0.0197
Epoch 9/50
14/14 [==============================] - 0s 18ms/step - loss: 0.0235 - val_loss: 0.0189
Epoch 10/50
14/14 [==============================] - 0s 

Epoch 32/50
8/8 [==============================] - 0s 19ms/step - loss: 0.0160 - val_loss: 0.0204
Epoch 33/50
8/8 [==============================] - 0s 23ms/step - loss: 0.0166 - val_loss: 0.0184
Epoch 34/50
8/8 [==============================] - 0s 23ms/step - loss: 0.0163 - val_loss: 0.0197
Epoch 35/50
8/8 [==============================] - 0s 22ms/step - loss: 0.0174 - val_loss: 0.0193
Epoch 36/50
8/8 [==============================] - 0s 23ms/step - loss: 0.0178 - val_loss: 0.0189
Epoch 37/50
8/8 [==============================] - 0s 24ms/step - loss: 0.0168 - val_loss: 0.0201
Epoch 38/50
8/8 [==============================] - 0s 23ms/step - loss: 0.0166 - val_loss: 0.0188
Epoch 39/50
8/8 [==============================] - 0s 23ms/step - loss: 0.0181 - val_loss: 0.0195
Epoch 40/50
8/8 [==============================] - 0s 22ms/step - loss: 0.0172 - val_loss: 0.0196
Epoch 41/50
8/8 [==============================] - 0s 21ms/step - loss: 0.0160 - val_loss: 0.0188
Epoch 42/50
8/8 [===

7/7 [==============================] - 0s 18ms/step - loss: 0.0544 - val_loss: 0.0451
Epoch 14/50
7/7 [==============================] - 0s 21ms/step - loss: 0.0512 - val_loss: 0.0455
Epoch 15/50
7/7 [==============================] - 0s 18ms/step - loss: 0.0534 - val_loss: 0.0463
Epoch 16/50
7/7 [==============================] - 0s 19ms/step - loss: 0.0507 - val_loss: 0.0447
Epoch 17/50
7/7 [==============================] - 0s 19ms/step - loss: 0.0550 - val_loss: 0.0446
Epoch 18/50
7/7 [==============================] - 0s 18ms/step - loss: 0.0508 - val_loss: 0.0458
Epoch 19/50
7/7 [==============================] - 0s 18ms/step - loss: 0.0505 - val_loss: 0.0444
Epoch 20/50
7/7 [==============================] - 0s 18ms/step - loss: 0.0549 - val_loss: 0.0446
Epoch 21/50
7/7 [==============================] - 0s 55ms/step - loss: 0.0537 - val_loss: 0.0435
Epoch 22/50
7/7 [==============================] - 0s 27ms/step - loss: 0.0531 - val_loss: 0.0466
Epoch 23/50
7/7 [===============

11/11 [==============================] - 0s 25ms/step - loss: 0.0412 - val_loss: 0.0382
Epoch 46/50
11/11 [==============================] - 0s 18ms/step - loss: 0.0402 - val_loss: 0.0380
Epoch 47/50
11/11 [==============================] - 0s 18ms/step - loss: 0.0400 - val_loss: 0.0403
Epoch 48/50
11/11 [==============================] - 0s 16ms/step - loss: 0.0398 - val_loss: 0.0389
Epoch 49/50
11/11 [==============================] - 0s 18ms/step - loss: 0.0390 - val_loss: 0.0388
Epoch 50/50
2/2 [==============================] - 1s 6ms/step
Epoch 1/50
 1/11 [=>............................] - ETA: 1:13 - loss: 0.3566